# Parkinson Disease Detection - Paper-Like Reproduction

This notebook trains the 4 transfer learning models used in the paper `Utilizing deep learning models in an intelligent spiral drawing classification system for Parkinson's disease classification`.

Goal: keep the training setup as close as possible to the paper, not to optimize the score.

Important limitations:
- The paper does not publish the exact train/test file names or random seed.
- The paper tables report different test supports: 18 samples for VGG19/InceptionV3 and 20 samples for ResNet50V2/DenseNet169.
- This notebook uses one fixed stratified split with 20 test samples because 20 is closest to an 80/20 split of 102 images and matches two paper result tables.
- Exact paper accuracy may still not be reproduced.


## 1. Setup

In [ ]:
import os
import random
import gc

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from tensorflow.keras import Model
from tensorflow.keras.applications import DenseNet169, InceptionV3, ResNet50V2, VGG19
from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess
from tensorflow.keras.applications.inception_v3 import preprocess_input as inception_preprocess
from tensorflow.keras.applications.resnet_v2 import preprocess_input as resnetv2_preprocess
from tensorflow.keras.applications.vgg19 import preprocess_input as vgg19_preprocess
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.layers import Dense
from tensorflow.keras.metrics import AUC
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print('TensorFlow version:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted.')
except Exception as exc:
    print('Google Drive mount skipped:', exc)

## 2. Configuration

In [ ]:
# Dataset folder should contain the Kaggle structure:
# drawings/spiral/training/healthy
# drawings/spiral/training/parkinson
# drawings/spiral/testing/healthy
# drawings/spiral/testing/parkinson
DATASET_PATH = '/content/drive/MyDrive/Parkinsons/drawings/spiral'
SAVE_PATH = '/content/drive/MyDrive/Parkinson_Models_Paper_Reproduction'

RANDOM_SEED = 42
EPOCHS = 50
LEARNING_RATE = 1e-4
VAL_SIZE_FROM_TRAIN = 0.2

# The paper states an 80/20 split. With 102 images, this is about 20 test images.
# Using an integer 20 gives exactly 10 healthy and 10 parkinson test images with stratification.
TEST_SIZE = 20

# Paper text says pixel values are normalized to [0, 1].
# Set this to True only if you want Keras ImageNet preprocessing instead.
USE_KERAS_PREPROCESS = False

# The paper does not mention early stopping or threshold tuning.
# Keep final epoch weights for the closest paper-like behavior.
SAVE_BEST_VAL_CHECKPOINT = True
EVALUATE_BEST_VAL_MODEL = False

os.makedirs(SAVE_PATH, exist_ok=True)
os.makedirs(os.path.join(SAVE_PATH, 'splits'), exist_ok=True)
os.makedirs(os.path.join(SAVE_PATH, 'plots'), exist_ok=True)
os.makedirs(os.path.join(SAVE_PATH, 'models'), exist_ok=True)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

print('Dataset path:', DATASET_PATH)
print('Save path:', SAVE_PATH)
print('Epochs:', EPOCHS)
print('Test size:', TEST_SIZE)
print('Use Keras preprocess:', USE_KERAS_PREPROCESS)

## 3. Load Dataset And Create Paper-Like Split

In [ ]:
IMAGE_EXTENSIONS = ('.png', '.jpg', '.jpeg', '.bmp')

def collect_spiral_dataset(dataset_root):
    rows = []
    for split_name in ['training', 'testing', '']:
        split_dir = os.path.join(dataset_root, split_name) if split_name else dataset_root
        for label in ['healthy', 'parkinson']:
            label_dir = os.path.join(split_dir, label)
            if not os.path.isdir(label_dir):
                continue
            for filename in sorted(os.listdir(label_dir)):
                if not filename.lower().endswith(IMAGE_EXTENSIONS):
                    continue
                file_path = os.path.join(label_dir, filename)
                if os.path.isfile(file_path):
                    rows.append({'filepath': file_path, 'label': label, 'source_split': split_name or 'root'})
    df = pd.DataFrame(rows).drop_duplicates(subset=['filepath']).reset_index(drop=True)
    if df.empty:
        raise FileNotFoundError(f'No spiral images found in {dataset_root}')
    return df

all_df = collect_spiral_dataset(DATASET_PATH)

print('Total images:', len(all_df))
print(all_df['label'].value_counts())
print('Source split counts:')
print(all_df.groupby(['source_split', 'label']).size())

if len(all_df) != 102:
    print('Warning: the paper dataset has 102 spiral images. Your collected count is', len(all_df))

In [ ]:
train_val_df, test_df = train_test_split(
    all_df,
    test_size=TEST_SIZE,
    stratify=all_df['label'],
    random_state=RANDOM_SEED
)

train_df, val_df = train_test_split(
    train_val_df,
    test_size=VAL_SIZE_FROM_TRAIN,
    stratify=train_val_df['label'],
    random_state=RANDOM_SEED
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

train_df.to_csv(os.path.join(SAVE_PATH, 'splits', 'train_split.csv'), index=False)
val_df.to_csv(os.path.join(SAVE_PATH, 'splits', 'val_split.csv'), index=False)
test_df.to_csv(os.path.join(SAVE_PATH, 'splits', 'test_split.csv'), index=False)

print('Train samples:', len(train_df))
print(train_df['label'].value_counts())
print('Validation samples:', len(val_df))
print(val_df['label'].value_counts())
print('Test samples:', len(test_df))
print(test_df['label'].value_counts())

## 4. Paper Model Configurations

In [ ]:
MODEL_CONFIGS = {
    'VGG19': {
        'builder': VGG19,
        'preprocess': vgg19_preprocess,
        'img_size': 100,
        'batch_size': 16,
        'dense_units': 64,
        'paper_accuracy': 0.72,
    },
    'InceptionV3': {
        'builder': InceptionV3,
        'preprocess': inception_preprocess,
        'img_size': 100,
        'batch_size': 32,
        'dense_units': 128,
        'paper_accuracy': 0.89,
    },
    'ResNet50V2': {
        'builder': ResNet50V2,
        'preprocess': resnetv2_preprocess,
        'img_size': 224,
        'batch_size': 32,
        'dense_units': 128,
        'paper_accuracy': 0.80,
    },
    'DenseNet169': {
        'builder': DenseNet169,
        'preprocess': densenet_preprocess,
        'img_size': 224,
        'batch_size': 16,
        'dense_units': 128,
        'paper_accuracy': 0.85,
    },
}

for model_name, cfg in MODEL_CONFIGS.items():
    print(f"{model_name}: img={cfg['img_size']}, batch={cfg['batch_size']}, dense={cfg['dense_units']}, paper_acc={cfg['paper_accuracy'] * 100:.0f}%")

## 5. Training Helpers

In [ ]:
def make_datagen(config, augment=False):
    preprocess_fn = config['preprocess'] if USE_KERAS_PREPROCESS else None
    rescale = None if USE_KERAS_PREPROCESS else 1.0 / 255.0

    if augment:
        return ImageDataGenerator(
            preprocessing_function=preprocess_fn,
            rescale=rescale,
            rotation_range=20,
            width_shift_range=0.2,
            height_shift_range=0.2,
            horizontal_flip=True,
            fill_mode='nearest'
        )

    return ImageDataGenerator(preprocessing_function=preprocess_fn, rescale=rescale)


def create_generators(model_name):
    config = MODEL_CONFIGS[model_name]
    target_size = (config['img_size'], config['img_size'])

    train_datagen = make_datagen(config, augment=True)
    eval_datagen = make_datagen(config, augment=False)

    train_generator = train_datagen.flow_from_dataframe(
        train_df,
        x_col='filepath',
        y_col='label',
        target_size=target_size,
        batch_size=config['batch_size'],
        class_mode='categorical',
        shuffle=True,
        seed=RANDOM_SEED
    )

    val_generator = eval_datagen.flow_from_dataframe(
        val_df,
        x_col='filepath',
        y_col='label',
        target_size=target_size,
        batch_size=config['batch_size'],
        class_mode='categorical',
        shuffle=False
    )

    test_generator = eval_datagen.flow_from_dataframe(
        test_df,
        x_col='filepath',
        y_col='label',
        target_size=target_size,
        batch_size=config['batch_size'],
        class_mode='categorical',
        shuffle=False
    )

    return train_generator, val_generator, test_generator


def create_paper_model(model_name):
    config = MODEL_CONFIGS[model_name]
    base_model = config['builder'](
        weights='imagenet',
        include_top=False,
        pooling='avg',
        input_shape=(config['img_size'], config['img_size'], 3)
    )

    # The paper describes the pretrained model as a feature extractor.
    base_model.trainable = False

    x = base_model.output
    x = Dense(config['dense_units'], activation='relu')(x)
    outputs = Dense(2, activation='softmax')(x)

    model = Model(inputs=base_model.input, outputs=outputs, name=f'{model_name}_paper_like')
    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss='categorical_crossentropy',
        metrics=['accuracy', AUC(name='auc')]
    )

    print(f'{model_name} created')
    print('Total params:', f'{model.count_params():,}')
    print('Trainable params:', f'{sum(tf.size(w).numpy() for w in model.trainable_weights):,}')
    return model


def plot_training_history(history, model_name):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].plot(history.history['accuracy'], label='Training Accuracy')
    axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy')
    axes[0].set_title(f'{model_name} Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(history.history['loss'], label='Training Loss')
    axes[1].plot(history.history['val_loss'], label='Validation Loss')
    axes[1].set_title(f'{model_name} Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    axes[2].plot(history.history['auc'], label='Training AUC')
    axes[2].plot(history.history['val_auc'], label='Validation AUC')
    axes[2].set_title(f'{model_name} AUC')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('AUC')
    axes[2].legend()
    axes[2].grid(alpha=0.3)

    plt.tight_layout()
    plot_path = os.path.join(SAVE_PATH, 'plots', f'{model_name}_history.png')
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved:', plot_path)


def evaluate_model(model, test_generator, model_name):
    test_generator.reset()
    y_prob = model.predict(test_generator, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)
    y_true = test_generator.classes

    class_indices = test_generator.class_indices
    target_names = [label for label, _ in sorted(class_indices.items(), key=lambda item: item[1])]
    parkinson_idx = class_indices['parkinson']

    test_loss, test_acc, keras_auc = model.evaluate(test_generator, verbose=0)
    sklearn_acc = accuracy_score(y_true, y_pred)
    test_auc = roc_auc_score(y_true, y_prob[:, parkinson_idx])

    print(f'Classification report - {model_name}')
    print(classification_report(y_true, y_pred, target_names=target_names, digits=4, zero_division=0))

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
    plt.title(f'{model_name} Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    cm_path = os.path.join(SAVE_PATH, 'plots', f'{model_name}_confusion_matrix.png')
    plt.savefig(cm_path, dpi=150, bbox_inches='tight')
    plt.show()

    fpr, tpr, _ = roc_curve(y_true, y_prob[:, parkinson_idx])
    plt.figure(figsize=(7, 6))
    plt.plot(fpr, tpr, label=f'ROC curve (area = {test_auc:.2f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'{model_name} ROC')
    plt.legend(loc='lower right')
    roc_path = os.path.join(SAVE_PATH, 'plots', f'{model_name}_roc.png')
    plt.savefig(roc_path, dpi=150, bbox_inches='tight')
    plt.show()

    print('Saved:', cm_path)
    print('Saved:', roc_path)

    return {
        'test_loss': float(test_loss),
        'test_acc': float(sklearn_acc),
        'keras_test_acc': float(test_acc),
        'keras_auc': float(keras_auc),
        'test_auc': float(test_auc),
        'confusion_matrix': cm,
        'y_true': y_true,
        'y_pred': y_pred,
        'y_prob': y_prob,
    }

## 6. Train The 4 Paper Models

In [ ]:
def run_experiment(model_name):
    tf.keras.backend.clear_session()
    gc.collect()

    print('= ' * 40)
    print('Training:', model_name)
    print('= ' * 40)

    train_generator, val_generator, test_generator = create_generators(model_name)
    print('Train batches:', len(train_generator))
    print('Validation batches:', len(val_generator))
    print('Test batches:', len(test_generator))

    model = create_paper_model(model_name)
    callbacks = []
    checkpoint_path = os.path.join(SAVE_PATH, 'models', f'{model_name}_best_val.keras')

    if SAVE_BEST_VAL_CHECKPOINT:
        callbacks.append(ModelCheckpoint(
            checkpoint_path,
            monitor='val_accuracy',
            mode='max',
            save_best_only=True,
            verbose=1
        ))

    history = model.fit(
        train_generator,
        validation_data=val_generator,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1
    )

    plot_training_history(history, model_name)

    if EVALUATE_BEST_VAL_MODEL and os.path.exists(checkpoint_path):
        print('Loading best validation checkpoint for evaluation:', checkpoint_path)
        model = tf.keras.models.load_model(checkpoint_path)
    else:
        print('Evaluating final epoch model, matching the paper-like setting.')

    eval_results = evaluate_model(model, test_generator, model_name)

    final_model_path = os.path.join(SAVE_PATH, 'models', f'{model_name}_final.keras')
    model.save(final_model_path)
    print('Saved final model:', final_model_path)

    return {
        'model': model,
        'history': history,
        'eval': eval_results,
        'checkpoint_path': checkpoint_path,
        'final_model_path': final_model_path,
        'config': MODEL_CONFIGS[model_name],
    }


results = {}
for model_name in ['VGG19', 'InceptionV3', 'ResNet50V2', 'DenseNet169']:
    results[model_name] = run_experiment(model_name)

## 7. Compare With Paper Results

In [ ]:
summary_rows = []
for model_name, data in results.items():
    paper_acc = data['config']['paper_accuracy']
    test_acc = data['eval']['test_acc']
    test_auc = data['eval']['test_auc']
    summary_rows.append({
        'model': model_name,
        'paper_accuracy': paper_acc,
        'notebook_test_accuracy': test_acc,
        'difference': test_acc - paper_acc,
        'notebook_test_auc': test_auc,
        'test_samples': len(test_df),
        'img_size': data['config']['img_size'],
        'batch_size': data['config']['batch_size'],
        'dense_units': data['config']['dense_units'],
    })

summary_df = pd.DataFrame(summary_rows)
summary_path = os.path.join(SAVE_PATH, 'paper_like_results_summary.csv')
summary_df.to_csv(summary_path, index=False)

display(summary_df)
print('Saved:', summary_path)

plt.figure(figsize=(10, 5))
x = np.arange(len(summary_df))
width = 0.35
plt.bar(x - width / 2, summary_df['paper_accuracy'] * 100, width, label='Paper Accuracy')
plt.bar(x + width / 2, summary_df['notebook_test_accuracy'] * 100, width, label='Notebook Accuracy')
plt.xticks(x, summary_df['model'])
plt.ylabel('Accuracy (%)')
plt.title('Paper Accuracy vs Paper-Like Notebook Accuracy')
plt.ylim(0, 100)
plt.legend()
plt.grid(axis='y', alpha=0.3)
comparison_path = os.path.join(SAVE_PATH, 'plots', 'paper_vs_notebook_accuracy.png')
plt.savefig(comparison_path, dpi=150, bbox_inches='tight')
plt.show()
print('Saved:', comparison_path)

## 8. Reproducibility Notes For Report

Use these notes in your PBL report if the accuracy is different from the paper:

- This notebook follows the paper architecture table: input sizes, batch sizes, dense layer sizes, softmax output, categorical cross-entropy, Adam optimizer and 50 epochs.
- Exact reproduction is not guaranteed because the paper does not provide exact file split, random seed, package versions, or full source code.
- The paper's own test supports are inconsistent across models, so the exact evaluation set cannot be reconstructed from the article alone.
- The dataset is very small. With 20 test samples, one changed prediction changes accuracy by 5 percentage points.
- Colab GPU nondeterminism and TensorFlow/Keras version differences can also change results.
